# FirstSlip Phase 2: open models (Qwen) with vLLM on Colab

Runs the **same frozen prompt** as the API model on Qwen3.5-2B, Qwen3.5-4B and Qwen3.5-27B (FP8). All the logic is in `scripts/phase2_run_vllm.py`; this notebook only sets up the environment and calls it.

- Runtime: **Colab Pro, A100** (40 or 80 GB). Greedy decoding with thinking off (`config/phase2.json`).
- Each model runs in its **own process**, so its GPU memory is fully freed before the next one loads.
- **Step 1** (dev) is for freezing the prompt. Run **Step 2** (test grid) only after the prompt is frozen and committed.
- At the end, download `phase2_runs.zip` and unzip it into the repo's `runs/` folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/AbdulRehmanAntall/firstslip.git
%cd firstslip
!git log -1 --format='%H %s'

In [ ]:
# vLLM brings its own torch; the repo's pinned scientific stack covers the rest.
!pip install -q vllm
!pip install -q -r requirements.txt
import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__)

Regenerate the synthetic data (deterministic, seed `20261019`) and check it against the committed hashes.

In [ ]:
!python scripts/generate_synthetic.py > /dev/null
import hashlib, json, pathlib
man = json.loads(pathlib.Path('data/synthetic/manifest.json').read_text())
for split, meta in man['splits'].items():
    h = hashlib.sha256(pathlib.Path(f'data/synthetic/{split}.jsonl').read_bytes()).hexdigest()
    assert h == meta['sha256'], f'{split}: hash mismatch'
    print(split, 'OK')

## Step 1: dev runs (prompt freezing only)

If a Qwen3.5 model fails to load or runs out of memory, add `--fallback` for that model. The fallbacks are Qwen3-1.7B, Qwen3-4B and Qwen3.5-27B-GPTQ-Int4. Record any switch in PROGRESS.md.

In [ ]:
MODELS = ['qwen-2b', 'qwen-4b', 'qwen-27b']
for m in MODELS:
    !python scripts/phase2_run_vllm.py --grid dev --models {m}

## Step 2: full test grid (only after the prompt is frozen)

Pull first, then check that `PROMPT_VERSION` is the frozen one.

In [ ]:
!git pull -q && git log -1 --format='%H %s'
!grep PROMPT_VERSION core/prompts.py | head -1
for m in MODELS:
    !python scripts/phase2_run_vllm.py --grid test --models {m}

In [ ]:
!cd runs && zip -q ../phase2_runs.zip *_phase2_*qwen*.jsonl && cd .. && ls -la phase2_runs.zip
from google.colab import files; files.download('phase2_runs.zip')